# Selección de variables

Se diagnostica la tabla reconstruida y se prepara la evaluación temporal. Los rankings son descriptivos y no determinan una lista final. Los métodos implementados se prueban sobre ejemplos sintéticos; los experimentos predictivos sobre la planta requieren configuración aprobada.

## 1. Preparación

Se leen las copias intermedias locales sin reconstruir ni sobrescribir archivos. La tabla ingenierizada sirve para diagnóstico; los futuros pliegues parten de la tabla previa a limpieza.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
sys.path.insert(0, str(Path.cwd().parent / "src"))
from auxiliar_functions import tabla_simple, correlaciones_con_objetivo, parejas_muy_correlacionadas, SEMILLA
from preprocessing_pipeline import dividir_por_fecha
from limpieza import COLUMNAS_DE_CONTROL
from variables_interacciones import columnas_candidatas
from seleccion_de_variables import evaluar_temporal, aprender_seleccion, aplicar_seleccion

tabla = pd.read_pickle("../data/interim/tabla_completa.pkl")
ingenierizada = pd.read_pickle("../data/interim/seleccion_diagnostico.pkl")
train = dividir_por_fecha(ingenierizada)[0]
bruto_train = dividir_por_fecha(tabla)[0]
# La última etiqueta externa depende de un bloque de validación externa.
bruto_train = bruto_train.copy()
bruto_train.loc[bruto_train.index[-1], "objetivo"] = np.nan
candidatas = columnas_candidatas(train)
print("Tabla 1. Dimensiones verificadas localmente")
display(tabla_simple(pd.DataFrame({"Cantidad": {
    "Filas completas": len(tabla), "Columnas completas": tabla.shape[1],
    "Columnas ingenierizadas": ingenierizada.shape[1], "Filas entrenamiento": len(train),
    "Candidatas entrenamiento": len(candidatas),
    "Etiquetas utilizables dentro de entrenamiento externo": bruto_train.objetivo.notna().sum()
}}), precision=0))

Tabla 1. Dimensiones verificadas localmente


,Cantidad
Filas completas,1212
Columnas completas,453
Columnas ingenierizadas,337
Filas entrenamiento,848
Candidatas entrenamiento,323
Etiquetas utilizables dentro de entrenamiento externo,830


## 2. Inventario de candidatas

Se informa por columna la disponibilidad y la variabilidad únicamente en entrenamiento. Los infinitos se distinguen de los faltantes. El inventario conserva los nombres completos para poder revisar cualquier candidata antes de experimentar.

In [2]:
valores = train[candidatas].astype(float)
perfil = pd.DataFrame({"Tipo": train[candidatas].dtypes.astype(str),
    "Faltantes": valores.isna().sum(), "Infinitos": np.isinf(valores).sum(),
    "Distintos": valores.nunique()}).rename_axis("Variable").reset_index()
print("Tabla 2. Inventario completo de candidatas en entrenamiento")
with pd.option_context("display.max_rows", None):
    display(tabla_simple(perfil, precision=0))
constantes = [c for c in train.columns if c not in COLUMNAS_DE_CONTROL
              and pd.api.types.is_numeric_dtype(train[c]) and train[c].nunique() <= 1]
print("Tabla 3. Resumen de calidad")
display(tabla_simple(pd.DataFrame({"Cantidad": {
    "Columnas con faltantes": int((perfil.Faltantes > 0).sum()),
    "Celdas faltantes": int(perfil.Faltantes.sum()),
    "Celdas infinitas": int(perfil.Infinitos.sum()),
    "Columnas constantes fuera de controles": len(constantes)}}), precision=0))
print("Constantes:", constantes)

Tabla 2. Inventario completo de candidatas en entrenamiento


,Variable,Tipo,Faltantes,Infinitos,Distintos
0,Planta (Kw),float64,18,0,830
1,Elaboracion (Kw),float64,19,0,813
2,Bodega (Kw),float64,22,0,808
3,Cocina (Kw),float64,17,0,659
4,Envasado (Kw),float64,18,0,805
5,Linea 2 (Kw),float64,19,0,829
6,Linea 3 (Kw),float64,17,0,731
7,Linea 4 (Kw),float64,17,0,601
8,Servicios (Kw),float64,17,0,822
9,Sala Maq (Kw),float64,20,0,816


Tabla 3. Resumen de calidad


,Cantidad
Columnas con faltantes,319
Celdas faltantes,6107
Celdas infinitas,0
Columnas constantes fuera de controles,0


Constantes: []


## 3. Orden y disponibilidad de etiquetas

Se comprueba la identidad del objetivo con el consumo del bloque siguiente. Las diferencias de fechas se describen sin corregirlas. En cada frontera interna, la etiqueta de la fila anterior coincide con el consumo del bloque actual, disponible al cierre. Esto respalda gap cero para predicción diaria al cierre bajo la interpretación por bloques del equipo; no certifica la fecha física de cada bloque.

In [3]:
saltos = pd.to_datetime(bruto_train.DIA).diff().dt.days
print("Tabla 4. Transiciones de fechas dentro de entrenamiento")
display(tabla_simple(saltos.value_counts(dropna=False).sort_index().rename("Cantidad").rename_axis("Días entre etiquetas").reset_index(), precision=0))
identidad = tabla.objetivo.equals(tabla.consumo_dia.shift(-1))
fronteras = pd.DataFrame({
    "Bloque predictor": np.arange(len(train)-1),
    "Bloque de disponibilidad de etiqueta": np.arange(1,len(train)),
    "Bloque de origen de evaluación gap cero": np.arange(1,len(train)),
})
print("Tabla 5. Ejemplos de fronteras internas al cierre")
display(tabla_simple(fronteras.head(5), precision=0))
controles = {
    "Orden cronológico": bool(tabla.DIA.is_monotonic_increasing),
    "Índice único": bool(tabla.index.is_unique),
    "Sin nombres duplicados": bool(ingenierizada.columns.is_unique),
    "Objetivo corresponde al bloque siguiente": bool(identidad),
    "Sin infinitos en candidatas de entrenamiento": bool(not np.isinf(valores.to_numpy()).any()),
    "Fronteras internas disponibles al cierre": bool((fronteras.iloc[:,1] <= fronteras.iloc[:,2]).all()),
    "Última etiqueta externa excluida del evaluador": bool(pd.isna(bruto_train.objetivo.iloc[-1])),
}
print("Tabla 6. Controles de diagnóstico")
display(tabla_simple(pd.DataFrame({"Resultado": controles})))
assert all(controles.values()), "Revisar controles antes de cualquier experimento."

Tabla 4. Transiciones de fechas dentro de entrenamiento


,Días entre etiquetas,Cantidad
0,0,15
1,1,821
2,2,8
3,4,3
4,nan,1


Tabla 5. Ejemplos de fronteras internas al cierre


,Bloque predictor,Bloque de disponibilidad de etiqueta,Bloque de origen de evaluación gap cero
0,0,1,1
1,1,2,2
2,2,3,3
3,3,4,4
4,4,5,5


Tabla 6. Controles de diagnóstico


,Resultado
Orden cronológico,True
Índice único,True
Sin nombres duplicados,True
Objetivo corresponde al bloque siguiente,True
Sin infinitos en candidatas de entrenamiento,True
Fronteras internas disponibles al cierre,True
Última etiqueta externa excluida del evaluador,True


## 4. Rankings y redundancia descriptivos

Se contrastan Spearman y Pearson sin fijar cantidad de variables. Se omite la última etiqueta del entrenamiento externo. La tabla de pares utiliza el umbral descriptivo histórico del EDA y no elimina columnas ni propone un nuevo umbral de selección.

In [4]:
# Reaprender bases sin la etiqueta que procede de validación externa.
from limpieza import aprender_reglas_de_limpieza, limpiar
from preprocessing_pipeline import fitear_reglas_de_variables, transform_variables
limpio_seguro = limpiar(bruto_train, aprender_reglas_de_limpieza(bruto_train))
reglas_seguras = fitear_reglas_de_variables(limpio_seguro)
diagnostico_train = transform_variables(limpio_seguro, reglas_seguras).iloc[:-1].copy()
candidatas = columnas_candidatas(diagnostico_train)
ranking = correlaciones_con_objetivo(diagnostico_train, candidatas)
print("Tabla 7. Ranking preliminar completo, sin selección")
with pd.option_context("display.max_rows", None):
    display(tabla_simple(ranking.reset_index().rename(columns={"index":"Variable"}), precision=4))
pares = parejas_muy_correlacionadas(diagnostico_train, candidatas, umbral=0.9)
print("Tabla 8. Redundancia descriptiva en entrenamiento")
display(tabla_simple(pd.DataFrame({"Cantidad": {"Pares con Spearman absoluto al menos 0.9": len(pares)}}), precision=0))
display(tabla_simple(pares.head(20), precision=4))

Tabla 7. Ranking preliminar completo, sin selección


,Variable,Spearman,Pearson,Spearman absoluto
0,Frio (Kw),0.8468,0.8596,0.8468
1,Frio (Kw) x frio_hace_2_dias,0.8381,0.8282,0.8381
2,frio_media_7_dias,0.8334,0.8448,0.8334
3,Sala Maq (Kw),0.8328,0.7887,0.8328
4,Frio (Kw) x Consolidado Agua | Agua Planta (Hl),0.8283,0.8265,0.8283
5,Servicios (Kw),0.8178,0.8213,0.8178
6,Frio (Kw) x Totalizadores Energia | KW Trafo 9,0.8079,0.8096,0.8079
7,dia_anio_coseno,-0.8003,-0.8000,0.8003
8,frio_hace_2_dias x Consolidado Agua | Agua Planta (Hl),0.8001,0.7951,0.8001
9,dia_anio_coseno x Consolidado Agua | Agua Planta (Hl),-0.7866,-0.7640,0.7866


Tabla 8. Redundancia descriptiva en entrenamiento


,Cantidad
Pares con Spearman absoluto al menos 0.9,167


,Columna A,Columna B,Spearman,Pearson
0,KW Gral Planta,Totalizadores Energia | KW Gral Planta,1.0000,1.0000
1,Consolidado Produccion | Hl Cerveza L5,Totalizadores Produccion | Hl Cerveza L5,1.0000,1.0000
2,Consolidado Produccion | Cocimientos Diarios,Totalizadores Produccion | Cocimientos Diarios,1.0000,1.0000
3,Consolidado Produccion | Hl Cerveza L3,Totalizadores Produccion | Hl Cerveza L3,1.0000,1.0000
4,Consolidado Produccion | Hl Cerveza L4,Totalizadores Produccion | Hl Cerveza L4,1.0000,1.0000
5,Consolidado Produccion | Hl Cerveza L2,Totalizadores Produccion | Hl Cerveza L2,1.0000,1.0000
6,Consolidado KPI | Agua Planta de Agua/Hl,Consolidado KPI | Produccion Agua / Hl,-1.0000,-1.0000
7,Consolidado Agua | FC L1 y L2,Totalizadores Agua | FC L1 y L2,1.0000,1.0000
8,Consolidado Agua | Agua Cocina (Hl),Totalizadores Agua | FC Cocina,1.0000,1.0000
9,Consolidado Agua | Red L1 y L2,Totalizadores Agua | Red L1 y L2,1.0000,1.0000


## 5. Preparación de experimentos

El módulo admite Spearman, Pearson, información mutua para regresión, Lasso, Forward y Backward. La configuración, el estimador evaluador con su tratamiento de faltantes, los pliegues y el presupuesto deben aprobarse antes de ejecutar. La referencia sin selección usa configuracion=None. Cada llamada evalúa una configuración fija y no optimiza parámetros. Elegir configuraciones y reportar su mínimo en los mismos pliegues sería optimista; una búsqueda requerirá un nivel temporal adicional. Forward y Backward están implementados con búsqueda interna temporal desde datos previos a limpieza. Requieren candidatas, iniciales de Forward, cantidad objetivo, pasos, presupuesto de ajustes y mejora mínima explícitos, además de estimador y partición internos. Cada subconjunto respeta jerarquía antes de evaluarse. Si una columna no se reproduce en un pliegue, la búsqueda falla sin prefiltrarla. No se ejecutan aquí. RFE queda fuera de alcance.

In [5]:
configuracion_aprobada = None
particion_aprobada = None
estimador_aprobado = None
resultados_experimento = None
print("No se ejecutan experimentos predictivos sobre la planta.")
# Después de aprobar un experimento individual:
# resultados_experimento = evaluar_temporal(
#     bruto_train, particion_aprobada, estimador_aprobado, configuracion_aprobada)
# La estabilidad se informa como frecuencia de inclusión por variable entre pliegues,
# separando elección directa de incorporación por jerarquía.


No se ejecutan experimentos predictivos sobre la planta.


## 6. Revisión puntual de fronteras

La copia histórica recibió la última etiqueta de entrenamiento, correspondiente al primer consumo de validación externa. La comprobación local confirmó su uso y que quitarla no cambia las bases elegidas. El ranking de este notebook reaprende ahora las bases con esa etiqueta anulada, sin modificar la copia histórica. Las fechas repetidas o con saltos mantienen el objetivo por bloque, sin garantizar el día calendario físico. Gap cero representa emisión al cierre. Forward y Backward realizan búsqueda interna; la evaluación externa no participa en ella. El costo máximo se declara por búsqueda y se multiplica por el número de pliegues externos, más sus ajustes finales.

## 7. Conclusión

El diagnóstico se apoya en la reconstrucción local y mantiene intactos los datos originales. La tabla intermedia no reemplaza el reaprendizaje por pliegue. No se eligió un método ganador, una cantidad final ni un subconjunto definitivo. Falta aprobar el diseño de comparación, la configuración de cada método y el estimador auxiliar. La definición por bloques conserva la incertidumbre de las fechas rotuladas. El contrato de dos argumentos señala explícitamente la configuración pendiente hasta que se apruebe una política final.